# PARC2026 — Static Quality Analyzer V1

Dataset Factory の **V0 Raw → V1 Clean候補** を作るため、全episodeの state/action trajectory を静的解析します。

このV1で計測するもの: EEF移動量・path・jerk、action強度、idle比率、gripper switch、timestamp/frame integrity、task-relative outlier。

このV1で **推測しない** もの: task success、collision、replayability。これらは simulator state を使う Replay Validator の責務です。

## Self-contained preflight
`00_a100_preflight.ipynb` を別に実行していなくても、このNotebook単体でworkspaceとrepoを準備します。

In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, sys

print('python:', sys.version)
print('platform:', platform.platform())
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)

ROOT = Path('/content/parc2026')
for p in [ROOT, ROOT/'vendor', ROOT/'cache', ROOT/'datasets', ROOT/'outputs']:
    p.mkdir(parents=True, exist_ok=True)

REPO = ROOT / 'py_AI'
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/yu37330/py_AI.git', str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'], check=True)
print('repo:', REPO)
print('git:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())

## Dependencies
動画は不要です。Parquet trajectoryだけを読むため `huggingface_hub + pyarrow + pandas` を使います。

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub>=0.30', 'pyarrow>=16', 'pandas>=2', 'matplotlib>=3'], check=True)
import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download

## Datasetを解決
優先順は `PARC_DATASET_ROOT` → organizer combined → 公開 `Sylvest/libero_plus_lerobot` です。

公開fallbackでは **meta + data/**/*.parquet のみ** を取得し、videosはdownloadしません。20で取得済みのmetadataがあれば同じdirectoryへtrajectory dataだけ追加されます。

In [ ]:
PUBLIC_DATASET = 'Sylvest/libero_plus_lerobot'
ORGANIZER_ROOT = ROOT / 'datasets' / 'libero_combined_20hz'
PUBLIC_ROOT = ROOT / 'datasets' / 'public_libero_plus_metadata'
configured = os.environ.get('PARC_DATASET_ROOT')

def has_meta(p):
    return (p / 'meta' / 'info.json').exists()

def has_data(p):
    return any(p.glob('data/**/*.parquet'))

if configured and has_meta(Path(configured)) and has_data(Path(configured)):
    DATASET_ROOT = Path(configured)
    QUALITY_SOURCE = 'configured:' + str(DATASET_ROOT)
elif has_meta(ORGANIZER_ROOT) and has_data(ORGANIZER_ROOT):
    DATASET_ROOT = ORGANIZER_ROOT
    QUALITY_SOURCE = 'organizer:libero_combined_20hz'
else:
    print('Organizer trajectory data is not present in Colab; downloading public LIBERO-plus parquet only...')
    snapshot_download(
        repo_id=PUBLIC_DATASET,
        repo_type='dataset',
        local_dir=str(PUBLIC_ROOT),
        allow_patterns=['meta/info.json', 'meta/episodes.jsonl', 'meta/tasks.jsonl', 'data/**/*.parquet'],
    )
    DATASET_ROOT = PUBLIC_ROOT
    QUALITY_SOURCE = 'public:' + PUBLIC_DATASET

assert has_meta(DATASET_ROOT), f'meta/info.json missing: {DATASET_ROOT}'
assert has_data(DATASET_ROOT), f'data/**/*.parquet missing: {DATASET_ROOT}'
info = json.loads((DATASET_ROOT/'meta'/'info.json').read_text())
print('dataset root:', DATASET_ROOT)
print('quality source:', QUALITY_SOURCE)
print('episodes:', info.get('total_episodes'), 'frames:', info.get('total_frames'), 'fps:', info.get('fps'))

## Static Quality Analyzerを実行
public LIBERO-plusなら 14,347 episode / 約223万frame を解析します。episode parquetは小さいため動画解析より軽量です。

`robust-z > 5` は **REVIEW候補** を出すための初期値であり、Reject閾値ではありません。

In [ ]:
OUT = ROOT / 'outputs' / 'static_quality_v1'
OUT.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, str(REPO/'tools/data/static_quality_analyzer.py'),
    '--root', str(DATASET_ROOT),
    '--out', str(OUT),
    '--smooth-window', '5',
    '--robust-z-threshold', '5.0',
]
subprocess.run(cmd, check=True)
summary_path = OUT / 'static_quality_summary.json'
summary = json.loads(summary_path.read_text())
summary['quality_source'] = QUALITY_SOURCE
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n')
print('outputs:', OUT)

## 結果を見る
`quality_status_v1=REVIEW` は削除判定ではありません。task内分布から外れたepisodeを、人手動画確認やReplay Validatorへ優先投入するためのqueueです。

In [ ]:
metrics = pd.read_csv(OUT/'episode_quality_metrics.csv')
review = pd.read_csv(OUT/'quality_review_candidates.csv')
taskq = pd.read_csv(OUT/'task_quality_summary.csv')
display(pd.DataFrame([summary]))
print('review candidates:', len(review), '/', len(metrics), f'({len(review)/len(metrics):.2%})')
display(taskq.sort_values('review_candidate_rate', ascending=False).head(20))
display(review.sort_values(['quality_flag_count','episode_index'], ascending=[False,True]).head(50))

## 分布を可視化
まず全体分布とtask別のREVIEW率を見ます。jerkは20Hz差分由来でノイズに敏感なので、raw値ではなく5-frame smooth版を主に見ます。

In [ ]:
import matplotlib.pyplot as plt

for col in ['eef_path_m', 'rms_cart_jerk_smooth', 'idle_ratio', 'motion_action_rms']:
    plt.figure(figsize=(9, 4))
    metrics[col].replace([np.inf, -np.inf], np.nan).dropna().hist(bins=60)
    plt.xlabel(col)
    plt.ylabel('episode count')
    plt.title(col + ' distribution')
    plt.show()

top = taskq.sort_values('review_candidate_rate', ascending=False).head(20)
plt.figure(figsize=(10, 6))
plt.barh(top['task_name'], top['review_candidate_rate'])
plt.xlabel('review candidate rate')
plt.title('Top task-relative review rates')
plt.gca().invert_yaxis()
plt.show()

## V1 Clean候補へ渡すepisode list
現段階では自動削除せず、`OK` と `REVIEW` のIDを分離して保存します。次のPRでtaskごとの分布・動画spot-check・必要ならReplay結果を見て、V1 Cleanの採用ルールを固定します。

In [ ]:
ok_ids = metrics.loc[metrics['quality_status_v1'] == 'OK', 'episode_index'].astype(int).tolist()
review_ids = metrics.loc[metrics['quality_status_v1'] == 'REVIEW', 'episode_index'].astype(int).tolist()
(OUT/'ok_episode_ids.json').write_text(json.dumps(ok_ids) + '\n')
(OUT/'review_episode_ids.json').write_text(json.dumps(review_ids) + '\n')
print('OK:', len(ok_ids), 'REVIEW:', len(review_ids))
print('saved:', OUT/'ok_episode_ids.json', OUT/'review_episode_ids.json')

## Exit criteria / 次
このNotebookの完了条件は `episode_quality_metrics.csv` と `quality_review_candidates.csv` が全episode分生成され、task別REVIEW率を確認できることです。

次は結果を使って **V1 Cleanルール固定 → V0 Raw / V1 Clean / V2 sqrt-balanced cheap ablation** へ進みます。Run A固定前には同じAnalyzerを運営 `libero_combined_20hz` に再実行します。